In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm
import CRPS.CRPS as pscore


import multiprocessing as mp
mp.set_start_method('spawn')


import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


sys.path.append('../../../Evaluation/')
import conduct_evaluation
from normal_evaluation.quantile_regression_evaluation import *
from normal_evaluation.normal_evaluation import SampleOutcomes_Normal

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]


In [2]:
with open('./quantile_regression_models.pkl', 'rb') as f:
    quantile_regression_models = pickle.load(f)

In [3]:
with open('../../transformed_event_logs/artificial_start_end_2_test.pickle', 'rb') as f:
    test_data = pickle.load(f)

activity_count = [
    'DIAGNOSIS',
    'QUALITY_CONTROL',
    'REPAIR',
 ]

resource_count = [
    '1',
    'Clark',
    'Jane',
    'Joe',
    'Karsten',
]

ii1 = [
    'intercase_n_1__DIAGNOSIS',
    'intercase_n_1__QUALITY_CONTROL',
    'intercase_n_1__REPAIR',
]

ii3 = [
    'intercase_n_3__DIAGNOSIS',
    'intercase_n_3__DIAGNOSIS_REPAIR',
    'intercase_n_3__DIAGNOSIS_REPAIR_QUALITY_CONTROL'
]

In [4]:
n_processes = 32
batch_size = 16
N = 1000

In [5]:
test_data

,concept:name,lifecycle:transition_start,time:timestamp_start,org:resource,case:concept:name,id_start,lifecycle:transition_complete,time:timestamp_complete,id_complete,duration,...,Clark,Jane,Joe,Karsten,intercase_n_1__DIAGNOSIS,intercase_n_1__QUALITY_CONTROL,intercase_n_1__REPAIR,intercase_n_3__DIAGNOSIS,intercase_n_3__DIAGNOSIS_REPAIR,intercase_n_3__DIAGNOSIS_REPAIR_QUALITY_CONTROL
0,DIAGNOSIS,START,2020-01-01 13:43:28.469233+00:00,Joe,0,1,COMPLETE,2020-01-01 14:59:41.608354+00:00,2,0 days 01:16:13.139121,...,0,0,1,0,0,0,0,0,0,0
1,REPAIR,START,2020-01-01 14:59:41.608354+00:00,Jane,0,4,COMPLETE,2020-01-04 18:38:11.639016+00:00,5,3 days 03:38:30.030662,...,0,1,1,0,0,0,0,0,0,0
2,QUALITY_CONTROL,START,2020-01-04 18:38:11.639016+00:00,Jane,0,7,COMPLETE,2020-01-05 06:43:33.329583+00:00,8,0 days 12:05:21.690567,...,0,2,1,0,0,0,0,0,0,0
3,DIAGNOSIS,START,2020-01-01 19:33:48.900814+00:00,1,1,10,COMPLETE,2020-01-01 20:34:39.406353+00:00,11,0 days 01:00:50.505539,...,0,0,0,0,0,0,1,0,1,0
4,REPAIR,START,2020-01-01 20:34:39.406353+00:00,1,1,13,COMPLETE,2020-01-02 09:05:17.659036+00:00,14,0 days 12:30:38.252683,...,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2671,REPAIR,START,2022-06-30 06:02:40.709625+00:00,Jane,890,8014,COMPLETE,2022-06-30 11:11:14.473595+00:00,8015,0 days 05:08:33.763970,...,0,2,0,0,0,0,1,0,1,0
2672,QUALITY_CONTROL,START,2022-06-30 11:11:14.473595+00:00,Jane,890,8017,COMPLETE,2022-06-30 13:45:04.355487+00:00,8018,0 days 02:33:49.881892,...,0,3,0,0,0,0,1,0,1,0
2673,DIAGNOSIS,START,2022-06-30 22:25:05.097535+00:00,Karsten,891,8020,COMPLETE,2022-06-30 23:15:20.606227+00:00,8021,0 days 00:50:15.508692,...,0,0,0,1,0,0,0,0,0,0
2674,REPAIR,START,2022-06-30 23:15:20.606227+00:00,Jane,891,8023,COMPLETE,2022-07-01 07:28:48.982352+00:00,8024,0 days 08:13:28.376125,...,0,1,0,1,0,0,0,0,0,0


In [6]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['A'], SampleOutcomes_QuantileRegression_A, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.092884566103655532671362312')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(10170.403054982227)

In [9]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['R'], SampleOutcomes_QuantileRegression_R, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [10]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.7514023745569859571481113428')

In [11]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-0.7514023745569859571481113428')

In [12]:
np.mean(get_pscores(likelihoods_A))

np.float64(11198.133147975737)

In [13]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['AR'], SampleOutcomes_QuantileRegression_AR, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [14]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-1.149117645119725825147178471')

In [15]:
np.mean(get_pscores(likelihoods_A))

np.float64(10100.40653114014)

In [16]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARS'], SampleOutcomes_QuantileRegression_ARS, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [17]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-767.0388895589283146874333092')

In [18]:
np.mean(get_pscores(likelihoods_A))

np.float64(12699.18090428725)

In [19]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSAC'], SampleOutcomes_QuantileRegression_ARSAC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [20]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-717.1589074626834947443486948')

In [21]:
np.mean(get_pscores(likelihoods_A))

np.float64(12726.483031622962)

In [22]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSRC'], SampleOutcomes_QuantileRegression_ARSRC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [23]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-393.4010018243381454787568075')

In [24]:
np.mean(get_pscores(likelihoods_A))

np.float64(12322.487449708371)

In [25]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSACRC'], SampleOutcomes_QuantileRegression_ARSACRC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [26]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-315.5530209859999481828083965')

In [27]:
np.mean(get_pscores(likelihoods_A))

np.float64(12331.19347418088)

In [28]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSD'], SampleOutcomes_QuantileRegression_ARSD, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [29]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-334.5532816613882806920937714')

In [30]:
np.mean(get_pscores(likelihoods_A))

np.float64(11456.051687040173)

In [31]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRC'], SampleOutcomes_QuantileRegression_ARSDACRC, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [32]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-14.26468640740990343475547320')

In [33]:
np.mean(get_pscores(likelihoods_A))

np.float64(11187.883830906116)

In [34]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII1'],
                                                   SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)


In [35]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-13.06059182931310501450392180')

In [36]:
np.mean(get_pscores(likelihoods_A))

np.float64(11546.381910234879)

In [37]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDACRCII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDACRCII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [38]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-27.65326126826463188215207093')

In [39]:
np.mean(get_pscores(likelihoods_A))

np.float64(11513.31777236191)

In [40]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII1'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'inter_instance_column_names' : ii1
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [41]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-126.6953179641580221695989358')

In [42]:
np.mean(get_pscores(likelihoods_A))

np.float64(11357.790388370804)

In [43]:
evaluator_A = conduct_evaluation.ConductEvaluation(quantile_regression_models['ARSDII3'],
                                                   SampleOutcomes_QuantileRegression_ARSDII, {
                                                        'case_id_key' : 'case:concept:name',
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource',
                                                        'inter_instance_column_names' : ii3
                                                    },
                                     test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True)

In [44]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-173.1845831732024874552399415')

In [45]:
np.mean(get_pscores(likelihoods_A))

np.float64(11772.070255107594)